# CTGF calcium imaging_reporting dynamic range for all cells over days
By Debora Masini, 2024

This notebook processes deltaF/F measurements (inscopix) and reports neuronal activity dynamic range across recording sessions.

Segment loads the final traces for all detected&accepted cells, ignores the time column, and calculates descriptive statistics for each neuron (works column by column to avoid memory issues). A separate results folder is created, and a new CSV file is generated containing descriptive stats for every cell. The script then computes and prints an overall summary based on the distribution of cell means, including the number of neurons, the global mean, and the global SEM.

### Overview of Workflow
1. **Input Data**
 - Reads multiple session-level calcium imaging files:
 - Each file contains: Time (s) column. One column per accepted neuron (trace values; e.g., ΔF/F or z-score)
 - Each file represents one recording session (e.g., s1, s2, etc.).

2. **Per-Cell Descriptive Statistics**
 - For each neuron column: Computes (NaN-safe): Mean, SEM (standard error of mean), Median, Maximum, Minimum, Variance
 - Adds metadata: File → session filename, Cell → neuron column name
 - Appends results to combined output table. One row per (Session, Cell).

3. **Global Across-Session Summary**. Loads the combined table and computes:
 - Total number of neuron entries. Mean of per-cell means. SEM of per-cell means. Minimum variance across all cells. Maximum variance across all cells
 - Provides descriptive dynamic range characterization across sessions.

4. **Robust Central Tendency Analysis**
 - To reduce sensitivity to outliers: Computes median of per-cell medians. Computes SEM of per-cell medians.

5. **Session-Specific Subset Analysis** (Example: Day1 = Session 1)

6. **Final Output**
 - Cross-session descriptive statistics table. Dynamic range reporting base.


In [ ]:
import os
import csv
import pandas as pd

In [ ]:
directory = r'...'

output_folder = os.path.join(directory, 'results')
os.makedirs(output_folder, exist_ok=True)

output_file = os.path.join(output_folder, "cell_statistics_allfiles.csv")

# List subset files 
csv_files = [f for f in os.listdir(directory) if f.startswith('FINAL_') and f.endswith(".csv")]
print("Subset files found:")
for f in csv_files:
    print("  ", f)

# Create combined stats file
with open(output_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["File", "Cell", "Mean", "SEM", "Median", "Max", "Min", "Variance"])

#  Process each file
for file_name in csv_files:
    file_path = os.path.join(directory, file_name)    
    df = pd.read_csv(file_path)
    # Columns to drop
    drop = ["Time (s)"]
    cells = df.drop(columns=drop)
    # Compute stats per cell
    stats = pd.DataFrame({
        "Mean": cells.mean(skipna=True),
        "SEM": cells.sem(skipna=True),
        "Median": cells.median(skipna=True),
        "Max": cells.max(skipna=True),
        "Min": cells.min(skipna=True),
        "Variance": cells.var(skipna=True) 
    # Variance is a statistical measurement that shows how far each number in the array is spread out from the mean. In other words, it measures the dispersion or spread in the data.
    })

    stats["Cell"] = stats.index # Add cell names explicitly
    stats["File"] = file_name # Add file + day
    stats = stats[["File", "Cell", "Mean", "SEM", "Median", "Max", "Min", "Variance"]] # Reorder columns
    # Append properly
    with open(output_file, "a", newline="") as f:
        stats.to_csv(f, header=False, index=False)

#  Compute final combined summary (treating each neuron-session instance independently)
stats_table = pd.read_csv(output_file)
overall_mean = stats_table["Mean"].mean()
overall_sem = stats_table["Mean"].sem()
num_neurons = len(stats_table)
print("\nSummary across files:")
print(f"Total neurons (all files, same neuron can be represented in different days): {num_neurons}")
print(f"Combined stats saved to (all files, same neuron can be represented in different days): {output_file}")

"""
example print:
Summary across files:
Total neurons (all files, same neuron can be represented in different days): 925
"""

In [ ]:
#report min and max variance
stats_table = pd.read_csv(output_file)   # same path
min_variance = stats_table["Variance"].min()
max_variance = stats_table["Variance"].max()
print(f"Min variance (All days): {min_variance:.4f}")
print(f"Max variance (All days): {max_variance:.4f}")

"""
example print:
Min variance (All days): 36.7144
Max variance (All days): 126075.2691
"""

In [ ]:
# sanity check>> To compute (median of medians) and SEM of the medians
# extract per-cell medians
cell_medians = stats_table["Median"]
overall_median_of_medians = cell_medians.median()
sem_of_medians = cell_medians.sem()
#print(f"Median of medians (All days): {overall_median_of_medians:.4f}")
#print(f"SEM of medians (All days): {sem_of_medians:.4f}")

In [ ]:
# what happens in Day1, i.e., session 1 empty Open field
# Filter rows where the file name contains 'Day1'
day1_table = stats_table[stats_table["File"].str.contains("Day1", case=False, na=False)]

# Compute variance bounds
min_variance = day1_table["Variance"].min()
max_variance = day1_table["Variance"].max()
print(f"Min variance (Day1): {min_variance:.4f}")
print(f"Max variance (Day1): {max_variance:.4f}")

# extract per-cell medians
cell_medians = day1_table["Median"]

# overall statistics
overall_median_of_medians = cell_medians.median()
sem_of_medians = cell_medians.sem()
#print(f"Median of medians (Day1): {overall_median_of_medians:.4f}")
#print(f"SEM of medians(Day1): {sem_of_medians:.4f}")

"""
example print:
Min variance (Day1): 311.3694
Max variance (Day1): 80844.8048
"""

In [ ]:
# script end